# Diabetic Retinopathy Stage Detection Pipeline

This notebook serves as the main entry point to execute the Diabetic Retinopathy classification pipeline on Google Colab.

## 1. Setup and Clone Repo
Clone the repository and install the necessary requirements.

In [ ]:
# IMPORTANT: Replace the URL below with your actual repository URL before running
!git clone https://github.com/your-username/diabetic-retinopathy-stage-detection.git
%cd diabetic-retinopathy-stage-detection

!pip install -r requirements.txt

## 2. Kaggle Download
Downloads the APTOS 2019 Blindness Detection dataset using the Kaggle API.
*Make sure you have added your `KAGGLE_USERNAME` and `KAGGLE_KEY` in the Colab Secrets tab on the left.*

In [ ]:
from src.data import download_aptos_dataset
download_aptos_dataset()

## 3. Exploratory Data Analysis (EDA)
Analyzes the class distributions and image properties.

In [ ]:
from src.data import run_eda
run_eda()

## 4. Split
Performs a strict 70/15/15 stratified train/validation/test split to prevent data leakage.

In [ ]:
from src.data_loader import split_dataset
split_dataset()

## 5. Preprocessing
Executes cropping, CLAHE, and unsharp masking, caching the processed images to disk.

In [ ]:
import os
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.preprocess import process_and_cache_dataset, generate_comparison_figures
from glob import glob
import random

train_input = os.path.join(RAW_DATA_DIR, "train_images")
train_output = os.path.join(PROCESSED_DATA_DIR, "train_images")
process_and_cache_dataset(train_input, train_output)

test_input = os.path.join(RAW_DATA_DIR, "test_images")
test_output = os.path.join(PROCESSED_DATA_DIR, "test_images")
if os.path.exists(test_input):
    process_and_cache_dataset(test_input, test_output)

# Optional: Generate a few comparison figures to see the effect
sample_imgs = glob(os.path.join(train_input, "*.png"))
if len(sample_imgs) >= 3:
    random.seed(42)
    generate_comparison_figures(random.sample(sample_imgs, 3))

## 6. Augmentation and Imbalance
Visualizes the training data augmentations and the class weighting strategy used to combat imbalance.

In [ ]:
from src.augment import show_augmented_examples
from src.data import compute_class_weights_from_train, visualize_oversampling

show_augmented_examples()
class_weights = compute_class_weights_from_train()
visualize_oversampling()

## 7. Model Architecture
Constructs the backbone (EfficientNetB0) and displays the parameter shifts between training phases.

In [ ]:
from src.model import display_summary
display_summary()

## 8. Training
Executes the 2-phase training loop. 
*Note: Adjust epochs appropriately for final evaluation.*

In [ ]:
from src.train import train_pipeline

# Set epochs to a lower value for quick testing, or higher (15, 20) for final results
train_pipeline(epochs_phase1=10, epochs_phase2=15)

## 9. Evaluation
Evaluates the best model strictly on the untouched test set.

In [ ]:
from src.evaluate import evaluate_model
evaluate_model()

## 10. Error Analysis
Displays the generated summary of where the model failed and why, alongside Grad-CAM interpretations of incorrect predictions.

In [ ]:
from IPython.display import display, Markdown, Image
from src.config import RESULTS_DIR
import os

error_md = os.path.join(RESULTS_DIR, "evaluation", "error_analysis_summary.md")
if os.path.exists(error_md):
    with open(error_md, 'r') as f:
        display(Markdown(f.read()))
        
incorrect_cam = os.path.join(RESULTS_DIR, "evaluation", "gradcam_incorrect.png")
if os.path.exists(incorrect_cam):
    print("\n--- Grad-CAM: Incorrect Predictions ---")
    display(Image(incorrect_cam))

## 11. Sample Predictions
Displays Grad-CAM heatmaps for correctly classified test instances.

In [ ]:
correct_cam = os.path.join(RESULTS_DIR, "evaluation", "gradcam_correct.png")
if os.path.exists(correct_cam):
    print("\n--- Grad-CAM: Correct Predictions ---")
    display(Image(correct_cam))

## 12. Export Results
Zips the entire `results/` folder (including models, plots, logs, and CAMs) for download.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('results_archive', 'zip', 'results')
files.download('results_archive.zip')